# Psychohistory Integration: ERH + Dynamic Prediction

This notebook demonstrates the integration of psychohistory concepts into the Ethical Riemann Hypothesis (ERH) model, creating a dynamic, predictive framework for moral judgment systems.

## Overview

We extend the static ERH model to include:
1. **Temporal Dimension**: Tracking error evolution over time E(x,t)
2. **Agent-Based Modeling**: Multi-agent interactions and population dynamics
3. **Network Dynamics**: Social network structures and opinion propagation
4. **Fluid Models**: Continuous PDE-based error density evolution
5. **Meta-Monitoring**: Second Foundation-style adaptive correction

This creates a "psychohistory + ERH" hybrid model that can predict and adapt to future error patterns.


In [ ]:
import sys
import os
from pathlib import Path

# Robust path setup
def setup_paths():
    current_dir = Path(os.getcwd())
    if current_dir.name == 'notebooks':
        simulation_dir = str(current_dir.parent)
        if simulation_dir not in sys.path:
            sys.path.insert(0, simulation_dir)
        return simulation_dir
    elif current_dir.name == 'simulation':
        simulation_dir = str(current_dir)
        if simulation_dir not in sys.path:
            sys.path.insert(0, simulation_dir)
        return simulation_dir
    for parent in current_dir.parents:
        if parent.name == 'simulation':
            simulation_dir = str(parent)
            if simulation_dir not in sys.path:
                sys.path.insert(0, simulation_dir)
            return simulation_dir
    for path in ['..', '../simulation', 'simulation']:
        abs_path = os.path.abspath(path)
        if abs_path not in sys.path:
            sys.path.insert(0, abs_path)
    return None

setup_paths()

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Note: ipywidgets not available, interactive features disabled")

# Core imports
from core.action_space import generate_world
from core.judgement_system import BiasedJudge, NoisyJudge, ConservativeJudge, evaluate_judgement
from core.ethical_primes import select_ethical_primes, compute_Pi_and_error

# Psychohistory imports
from core.temporal_erh import (
    track_error_evolution, compute_Pi_temporal, compute_E_temporal,
    simulate_mule_effect, detect_mule_anomalies
)
from core.agent import EthicalAgent, AgentPopulation, SimpleEthicalAgent
from core.social_network import SocialNetwork
from core.meta_monitor import MetaMonitor
from core.abm_simulator import ABMSimulator
from core.hybrid_model import HybridPsychohistoryModel

# Analysis imports
from analysis.temporal_analysis import (
    analyze_temporal_trends, detect_anomalies, forecast_error_growth,
    compute_temporal_erh_satisfaction
)
from analysis.opinion_dynamics import degroot_model, hegselmann_krause_model, aggregate_beliefs
from analysis.fluid_model import solve_error_density_pde, fit_fluid_parameters, detect_critical_phenomena

# Visualization imports
from visualization.plots import setup_paper_style
from visualization.temporal_plots import (
    plot_3d_error_surface, plot_anomaly_timeline, plot_temporal_trends_comparison,
    plot_forecast_comparison
)
from visualization.network_plots import (
    plot_network_topology, plot_error_density_on_network, plot_centrality_comparison
)

setup_paper_style()
np.random.seed(42)

print("All modules imported successfully!")
print("Psychohistory integration ready!")


## Part 1: Temporal ERH - Time Series Analysis

First, we demonstrate temporal extension of ERH, tracking error evolution over time.


In [ ]:
# Configuration
num_actions = 1000
tau = 0.3
time_steps = 10
X_max = 100

# Generate action history (simulating time evolution)
actions_history = []
judge = BiasedJudge(bias_strength=0.2, noise_scale=0.1)

for t in range(time_steps):
    actions = generate_world(num_actions=num_actions, random_seed=42 + t)
    actions_history.append(actions)

print(f"Generated {time_steps} time steps of actions")
print(f"Each time step: {num_actions} actions")


In [ ]:
# Track error evolution
temporal_results = track_error_evolution(
    actions_history, judge, tau=tau, time_steps=time_steps, X_max=X_max
)

Pi_xt = temporal_results['Pi_xt']
B_xt = temporal_results['B_xt']
E_xt = temporal_results['E_xt']

print(f"Temporal ERH computed:")
print(f"  Pi_xt shape: {Pi_xt.shape}")
print(f"  E_xt shape: {E_xt.shape}")
print(f"  Time steps: {time_steps}, Complexity range: 1-{X_max}")

# Analyze temporal trends
x_values = np.arange(1, X_max + 1)
t_values = np.arange(time_steps)
trends = analyze_temporal_trends(E_xt, time_steps, x_values)

print(f"\nTemporal Trends:")
print(f"  Overall trend: {trends['overall_trend']['direction']}")
print(f"  Mean volatility: {trends['volatility']:.4f}")
print(f"  Mean error: {trends['mean_error']:.4f}")


In [ ]:
# Visualize 3D error surface
fig = plot_3d_error_surface(
    E_xt, x_values, t_values,
    title="Temporal Error Evolution E(x,t)",
    save_path='../output/figures/08_3d_error_surface.pdf',
    show=True
)


### Mule Effect Simulation

Simulate the "Mule effect" - sudden anomalies that disrupt predictions (analogous to Asimov's Mule).


In [ ]:
# Simulate Mule effect at time step 5
E_xt_mule = simulate_mule_effect(E_xt, mule_time=5, mule_strength=2.5, X_max=X_max)

# Detect anomalies
anomalies = detect_anomalies(E_xt_mule, method='combined', X_max=X_max)

print(f"Detected {len(anomalies)} anomalies")
if len(anomalies) > 0:
    print(f"\nTop 5 anomalies:")
    for i, anomaly in enumerate(anomalies[:5]):
        print(f"  {i+1}. t={anomaly['time']}, x={anomaly['complexity']}, "
              f"severity={anomaly['severity']}, violation={anomaly['violation_ratio']:.2f}x")

# Visualize anomalies
fig = plot_anomaly_timeline(
    anomalies, E_xt_mule, x_values, t_values,
    save_path='../output/figures/08_anomaly_timeline.pdf',
    show=True
)


### Error Forecasting

Forecast future error growth based on historical patterns.


In [ ]:
# Forecast error growth
forecast = forecast_error_growth(E_xt, forecast_horizon=5, method='linear', X_max=X_max)

print(f"Forecast completed:")
print(f"  Method: {forecast['method_used']}")
print(f"  Forecast horizon: {forecast['forecast_errors']['forecast_horizon']} steps")
print(f"  Mean forecast error: {forecast['forecast_errors']['mean_forecast']:.4f}")

# Visualize forecast
fig = plot_forecast_comparison(
    E_xt, forecast, x_values, t_values,
    selected_complexities=[10, 25, 50, 75],
    save_path='../output/figures/08_forecast_comparison.pdf',
    show=True
)


## Part 2: Agent-Based Modeling

Create a population of AI judgment agents and simulate their interactions.


In [ ]:
# Create ABM simulator
def judge_factory(i):
    """Create diverse judges for agents"""
    from core.judgement_system import BiasedJudge, NoisyJudge, ConservativeJudge
    judge_type = i % 3
    if judge_type == 0:
        return BiasedJudge(bias_strength=0.1 + 0.1 * (i % 5) / 5, name=f"Biased_{i}")
    elif judge_type == 1:
        return NoisyJudge(noise_scale=0.1 + 0.1 * (i % 5) / 5, name=f"Noisy_{i}")
    else:
        return ConservativeJudge(threshold=0.3 + 0.2 * (i % 5) / 5, name=f"Conservative_{i}")

abm = ABMSimulator(
    num_agents=50,
    judge_factory=judge_factory,
    network_topology='small_world',
    enable_meta_monitor=True
)

print(f"Created ABM simulator with {len(abm.population)} agents")
print(f"Network topology: {abm.network.get_network_statistics()}")


In [ ]:
# Run ABM simulation
abm_results = abm.run_simulation(
    num_time_steps=8,
    actions_per_step=500,
    tau=tau,
    X_max=X_max,
    track_erh=True,
    interaction_probability=0.15
)

print("ABM Simulation Complete!")
print(f"  Time steps: {abm.current_time + 1}")
print(f"  Population stats: {abm.population.get_population_statistics()}")

# Display ERH history
erh_history = abm_results['erh_history']
print(f"\nERH History Summary:")
for entry in erh_history[::2]:  # Every other time step
    analysis = entry.get('analysis', {})
    erh_sat = "SATISFIED" if analysis.get('erh_satisfied', False) else "NOT SATISFIED"
    exp = analysis.get('estimated_exponent', np.nan)
    print(f"  t={entry['time']}: ERH={erh_sat}, α={exp:.4f}, primes={entry.get('num_primes', 0)}")


## Part 3: Network Dynamics

Demonstrate opinion dynamics and belief aggregation through social networks.


In [ ]:
# Visualize network topology
fig = plot_network_topology(
    abm.network,
    node_color_attribute='error_rate',
    node_size_attribute='degree',
    save_path='../output/figures/08_network_topology.pdf',
    show=True
)


In [ ]:
# Run opinion dynamics (DeGroot model)
network = abm.network
agents = abm.population.agents

dynamics_result = degroot_model(agents, network, max_iterations=50)

print("Opinion Dynamics Results:")
print(f"  Converged: {dynamics_result['converged']}")
print(f"  Iterations: {dynamics_result['iterations']}")
print(f"  Final opinions range: [{min(dynamics_result['final_opinions']):.4f}, "
      f"{max(dynamics_result['final_opinions']):.4f}]")

# Aggregate beliefs
individual_errors = {agent.agent_id: agent.error_rate for agent in agents}
aggregated = aggregate_beliefs(individual_errors, network, dynamics_model='degroot')

print(f"\nBelief Aggregation:")
print(f"  Individual error range: [{min(individual_errors.values()):.4f}, "
      f"{max(individual_errors.values()):.4f}]")
print(f"  Aggregated range: [{min(aggregated.values()):.4f}, "
      f"{max(aggregated.values()):.4f}]")


## Part 4: Meta-Monitoring (Second Foundation)

Demonstrate the meta-layer monitoring system that automatically detects violations and adjusts parameters.


In [ ]:
# Get temporal ERH from ABM results
temporal_erh = abm.compute_temporal_erh(abm_results['actions_history'], tau=tau, X_max=X_max)
E_xt_abm = temporal_erh['E_xt']

# Monitor with meta-monitor
meta_monitor = MetaMonitor(violation_threshold=1.5, correction_enabled=True)

monitoring_results = []
for t in range(E_xt_abm.shape[0]):
    result = meta_monitor.monitor(E_xt_abm, t, X_max)
    monitoring_results.append(result)

# Summary
summary = meta_monitor.get_monitoring_summary()
print("Meta-Monitoring Summary:")
print(f"  Total violations: {summary['total_violations']}")
print(f"  Corrections applied: {summary['corrections_applied']}")
print(f"  Current parameters: C={summary['current_params']['C']:.4f}, "
      f"ε={summary['current_params']['epsilon']:.4f}")

if summary.get('severity_breakdown'):
    print(f"  Severity breakdown: {summary['severity_breakdown']}")


## Part 5: Fluid Model (Optional - Computationally Intensive)

Demonstrate continuous PDE-based error density evolution model.


In [ ]:
# Fit fluid parameters from observed data
fluid_params = fit_fluid_parameters(E_xt, x_values, t_values)

print("Fitted Fluid Model Parameters:")
print(f"  Convection velocity (v): {fluid_params['v']:.4f}")
print(f"  Diffusion coefficient (D): {fluid_params['D']:.4f}")
print(f"  Correction rate (α): {fluid_params['alpha']:.4f}")

# Solve fluid model (simplified for demonstration)
try:
    u_xt, x_grid, t_grid = solve_error_density_pde(
        x_range=(1, X_max),
        t_range=(0, time_steps),
        nx=min(30, X_max),  # Reduced for speed
        nt=min(20, time_steps),
        v=fluid_params['v'],
        D=fluid_params['D'],
        alpha=fluid_params['alpha']
    )
    
    # Detect critical phenomena
    critical_events = detect_critical_phenomena(u_xt, x_grid, t_grid, threshold=2.0)
    
    print(f"\nFluid Model Results:")
    print(f"  Solution shape: {u_xt.shape}")
    print(f"  Critical events detected: {len(critical_events)}")
    
    if len(critical_events) > 0:
        print(f"  First critical event: t={critical_events[0]['time_value']:.2f}, "
              f"x={critical_events[0]['complexity']:.2f}, severity={critical_events[0]['severity']}")
except Exception as e:
    print(f"Fluid model computation skipped: {e}")
    print("(This is normal if scipy sparse solvers have issues)")


## Part 6: Complete Hybrid Model

Run the complete integrated psychohistory model with all components.


In [ ]:
# Create hybrid model
hybrid = HybridPsychohistoryModel(
    num_agents=30,
    judge_factory=judge_factory,
    network_topology='small_world',
    enable_temporal=True,
    enable_network_dynamics=True,
    enable_fluid_model=False,  # Disable for speed
    enable_meta_monitor=True
)

print("Hybrid Model Created:")
summary = hybrid.get_summary()
print(f"  Agents: {summary['num_agents']}")
print(f"  Features: {summary['features_enabled']}")


In [ ]:
# Run complete hybrid simulation
print("Running hybrid simulation (this may take a moment)...")
hybrid_results = hybrid.run_simulation(
    num_time_steps=6,
    actions_per_step=500,
    tau=tau,
    X_max=X_max,
    network_dynamics_model='degroot'
)

print("\nHybrid Simulation Complete!")
print("=" * 60)

# Unified metrics
unified_metrics = hybrid.get_unified_metrics(hybrid_results)
print("\nUnified System Metrics:")
print(f"  ERH Satisfaction Rate: {unified_metrics['erh_satisfaction']['satisfaction_rate']:.2%}")
print(f"  Temporal Stability: {unified_metrics['temporal_stability']['trend_direction']}")
print(f"  Network Coherence: {'Converged' if unified_metrics['network_coherence']['converged'] else 'Not converged'}")
print(f"  System Health: {unified_metrics['system_health']['status']} "
      f"(score: {unified_metrics['system_health']['score']:.3f})")


### Adaptive Adjustment

Demonstrate adaptive parameter adjustment based on simulation results.


In [ ]:
# Perform adaptive adjustment
adjustments = hybrid.adaptive_adjustment(hybrid_results, target_exponent=0.5)

print("Adaptive Adjustments:")
if 'erh_parameters' in adjustments:
    params = adjustments['erh_parameters']
    print(f"  ERH Parameters: C={params['C']:.4f}, ε={params['epsilon']:.4f}")

if 'abm_calibration' in adjustments:
    calib = adjustments['abm_calibration']
    if calib.get('calibrated'):
        print(f"  ABM Calibration:")
        print(f"    Mean exponent: {calib['mean_exponent']:.4f}")
        print(f"    Satisfaction rate: {calib['satisfaction_rate']:.2%}")
        if calib.get('recommendations'):
            print(f"    Recommendations:")
            for rec in calib['recommendations']:
                print(f"      - {rec}")


## Summary and Conclusions

This notebook demonstrated the complete integration of psychohistory concepts into ERH:

1. **Temporal ERH**: Error evolution over time E(x,t)
2. **ABM Simulation**: Multi-agent interactions and population dynamics
3. **Network Dynamics**: Opinion propagation and belief aggregation
4. **Meta-Monitoring**: Automatic violation detection and parameter adjustment
5. **Hybrid Model**: Unified framework combining all components

The hybrid model provides a predictive, adaptive framework for analyzing moral judgment systems, analogous to Asimov's psychohistory but applied to ethical decision-making.
